# Alexandria: train the Hindi models on a free Colab T4

This notebook trains the four parts used by the app and saves them to Google Drive:

1. **Difficulty scorer**: 20 levels, from corpus word frequencies + grammar rules (CPU)
2. **Sentence generator**: LoRA fine-tune of Llama-3.2-3B on English-Hindi pairs (GPU)
3. **Translation evaluator**: MuRIL, scores translations 0-1 (GPU)
4. **Transliterator**: romanized Hindi → Devanagari (GPU)

**Before you start**
* Runtime → Change runtime type → **T4 GPU**.
* Create a Hugging Face token at https://huggingface.co/settings/tokens and add it in the Colab sidebar under 🔑 **Secrets** as `HF_TOKEN` (enable notebook access).
* With the same Hugging Face account, click *Agree* on these pages:
  [Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) (or set `PRESET = "qwen3-4b"` below to skip it),
  [BPCC](https://huggingface.co/datasets/ai4bharat/BPCC), [Samanantar](https://huggingface.co/datasets/ai4bharat/samanantar).
  Datasets you skip are left out automatically.

Every step writes to Drive and can be re-run: training steps resume from their last checkpoint after a disconnect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive/alexandria"
A = f"{ROOT}/artifacts"   # trained models (this is what the backend needs)
D = f"{ROOT}/data"        # prepared training data
os.makedirs(A, exist_ok=True); os.makedirs(D, exist_ok=True)
os.environ["ALEXANDRIA_CACHE"] = f"{ROOT}/cache"  # downloaded corpora survive restarts

PRESET = "llama3.2-3b"   # or "qwen3-4b" (no sign-up), "llama3.2-1b" (fastest); see models/README.md

In [ ]:
# Get the code. For a private repo, use a GitHub token:  https://<TOKEN>@github.com/SIGMobile-UIUC/alexandria.git
BRANCH = "main"
!test -d /content/alexandria || git clone -b {BRANCH} https://github.com/SIGMobile-UIUC/alexandria.git /content/alexandria
%cd /content/alexandria
!git pull -q
!pip install -q -e "models[train,serve]"
# Colab ships an old torchao that peft refuses to import; we don't use it.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("No HF_TOKEN secret found; gated models/datasets will be skipped:", e)

import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - switch the runtime to T4")

## 1. Difficulty scorer (CPU, roughly 30-60 min)
Counts word frequencies over Hindi corpora and fits the 20 level thresholds on the parallel data.
Look at the log at the end: it prints an example sentence for every level, which is how you check the levels make sense.

In [ ]:
!python -m alexandria_models.difficulty.build --output_dir {A}/difficulty

In [ ]:
import json
s = json.load(open(f"{A}/difficulty/scorer.json"))
for lv in s["level_summaries"]:
    print(f"level {lv['level']:2d} ({lv['mean_words']:.0f} words, {', '.join(lv['top_constructions'][:3])}):  {lv['examples'][0] if lv['examples'] else ''}")

## 2. Sentence generator
First the data: every parallel pair is scored and assigned a level; up to 3000 pairs per level are kept.

In [ ]:
!python -m alexandria_models.generator.prepare_data --difficulty_dir {A}/difficulty --output_dir {D}/generator
!cat {D}/generator/data_config.json

**Smoke test (5 min):** 50 steps to check memory and that the loss goes down. Watch for `nan`.

In [ ]:
!python -m alexandria_models.generator.train --data_dir {D}/generator --output_dir /content/gen_smoke \
    --preset {PRESET} --max_steps 50 --eval_steps 25 --logging_steps 5 --no_merge
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

**Full run** (a few hours for the 3B model). If Colab disconnects, re-run this cell: `--resume` continues from the last checkpoint on Drive.
At the end it merges the LoRA adapter into the base model (`generator/merged/`).

In [ ]:
!python -m alexandria_models.generator.train --data_dir {D}/generator --output_dir {A}/generator \
    --preset {PRESET} --resume

## 3. Translation evaluator (~30 min)
Human quality scores from WMT23 (English→Hindi) plus synthetic learner mistakes in both directions.

In [ ]:
!python -m alexandria_models.evaluator.prepare_data --output_dir {D}/evaluator
!python -m alexandria_models.evaluator.train --data_dir {D}/evaluator --output_dir {A}/evaluator --resume
!cat {A}/evaluator/evaluator_config.json

## 4. Transliterator (~30 min)
Aksharantar + Dakshina word pairs. Validation accuracy is printed after every epoch. For reference, the best published systems reach about 60% on Dakshina.

In [ ]:
!python -m alexandria_models.transliterator.prepare_data --output_dir {D}/transliterator
!python -m alexandria_models.transliterator.train --data_dir {D}/transliterator --output_dir {A}/transliterator

## 5. Pre-generate sentences (recommended)
Lets the backend serve generated sentences without running the LLM on a CPU server.

In [ ]:
!python -m alexandria_models.generator.generate --generator_dir {A}/generator --difficulty_dir {A}/difficulty \
    --evaluator_dir {A}/evaluator --per_level 200 --out {A}/generator/generated_bank.jsonl

## 6. Try everything

In [ ]:
from alexandria_models.inference import AlexandriaModels
m = AlexandriaModels.load(A)
print(m.status())
for lv in (1, 5, 10, 15, 20):
    for s in m.sentences(level=lv, n=2, mode="generate"):
        print(lv, s["hindi"], "|", s["english"], "| measured level", s.get("level"))
print(m.evaluate("मैं घर जा रहा हूँ।", "I am going home"))
print(m.evaluate("I am going home.", "main ghar ja raha hoon"))
print(m.evaluate("I am going home.", "main ghar nahi ja raha hoon"))   # negation flipped: should score low
print(m.transliterate("kya aap mujhe paani de sakte hain", k=3))
print(m.difficulty("यदि समय पर उपचार किया जाता तो रोगी की जान बच सकती थी।"))

## 7. Download the models
The backend needs the `artifacts/` folder. The generator's `merged/` folder is large (~6 GB for 3B), so leave it out if the server only uses the sentence banks.

In [ ]:
!cd {ROOT} && zip -qr /content/alexandria_artifacts_small.zip artifacts -x "artifacts/generator/merged/*" "artifacts/generator/adapter/*" "artifacts/*/checkpoints/*"
!ls -lh /content/alexandria_artifacts_small.zip
from google.colab import files
files.download("/content/alexandria_artifacts_small.zip")